# pythscribe — a simple `@wasm` demo

Write an **ordinary Python function**, put `@wasm` on top, and **call it** — it runs as a
sandboxed **WebAssembly** module, **bit-for-bit identical to CPython**.

- No build step, no `import kernels`, no plumbing. The **first call compiles** the function
  (once, cached on disk by its source); every call after is instant.
- The only `@wasm` import is `from pythscribe import wasm`. (NumPy shows up later only as the
  data buffers for the array kernels — the headline there is "== NumPy, bit-for-bit".)
- If the compiler or wasmtime isn't available, the very same function just runs as **plain
  Python** — nothing to change.

Five use cases below: **speed**, **determinism**, **the sandbox**, **1-D arrays**, **2-D / image**.
The same `@wasm` function runs unchanged in a notebook, on the server, in the browser, and inside
a Gradio / Streamlit app (last section).

## 1. Speed on non-vectorizable loops

`edit_distance` is a dynamic-programming loop NumPy can't vectorize. Just decorate it — that's the
whole API. A `@wasm` kernel is ordinary Python with a few fast-path rules (annotated arguments, a
scalar return, simple statements: no tuple-unpacking `a, b = ...`); step outside them and it just
keeps running as plain Python.

In [1]:
from pythscribe import wasm, binding_of

@wasm
def edit_distance(a: list[int], b: list[int]) -> int:
    n = len(a)
    m = len(b)
    prev = [0] * (m + 1)
    cur = [0] * (m + 1)
    for j in range(m + 1):
        prev[j] = j
    for i in range(1, n + 1):
        cur[0] = i
        for j in range(1, m + 1):
            cost = 1
            if a[i - 1] == b[j - 1]:
                cost = 0
            best = prev[j] + 1
            if cur[j - 1] + 1 < best:
                best = cur[j - 1] + 1
            if prev[j - 1] + cost < best:
                best = prev[j - 1] + cost
            cur[j] = best
        for j in range(m + 1):
            prev[j] = cur[j]
    return prev[m]

The first call compiles it to WASM. `binding_of(fn).mode` says how it ran: `server` = the compiled
`.wasm` ran in-process under wasmtime; `fallback` = plain Python (same answer).

In [2]:
print("before first call, mode:", binding_of(edit_distance).mode)   # fallback: nothing compiled yet

d = edit_distance([1, 2, 3, 4, 5], [1, 0, 3, 4, 9])

print("result:", d)
print("after  first call, mode:", binding_of(edit_distance).mode)   # server: it ran as WASM

before first call, mode: fallback


result: 2
after  first call, mode: server


In [3]:
import time, random

def edit_distance_py(a, b):
    n, m = len(a), len(b)
    prev = list(range(m + 1))
    cur = [0] * (m + 1)
    for i in range(1, n + 1):
        cur[0] = i
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev, cur = cur, prev
    return prev[m]

random.seed(0)
a  = [random.randint(0, 25) for _ in range(400)]
bb = [random.randint(0, 25) for _ in range(400)]
assert edit_distance(a, bb) == edit_distance_py(a, bb)   # @wasm == plain Python

def avg_ms(fn, n=20):
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    return (time.perf_counter() - t0) / n * 1e3

t_wasm = avg_ms(lambda: edit_distance(a, bb))
t_py   = avg_ms(lambda: edit_distance_py(a, bb))
print(f"@wasm {t_wasm:6.2f} ms | plain Python {t_py:6.2f} ms | speedup x{t_py / t_wasm:.1f}")

@wasm   3.91 ms | plain Python  64.63 ms | speedup x16.5


## 2. Bit-for-bit determinism (floats)

A float kernel — the `@wasm` result is **exactly** equal to CPython, not just close. That fixed
float behaviour is why "same bits on the server, in the browser, and in CPython" holds.

In [4]:
@wasm
def pairwise_abs_sum(xs: list[float]) -> float:
    n = len(xs)
    total = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = xs[i] - xs[j]
            if d < 0.0:
                d = -d
            total = total + d
    return total

def pairwise_abs_sum_py(xs):
    n = len(xs)
    total = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            d = xs[i] - xs[j]
            if d < 0.0:
                d = -d
            total = total + d
    return total

random.seed(1)
xs = [random.random() for _ in range(600)]
assert pairwise_abs_sum(xs) == pairwise_abs_sum_py(xs)   # exact, not approximate
print("@wasm =", repr(pairwise_abs_sum(xs)))
print("plain =", repr(pairwise_abs_sum_py(xs)), " -> bit-for-bit identical")
print(f"speedup x{avg_ms(lambda: pairwise_abs_sum_py(xs)) / avg_ms(lambda: pairwise_abs_sum(xs)):.1f}")

@wasm = 59813.3870826163
plain = 59813.3870826163  -> bit-for-bit identical


speedup x14.7


## 3. Run untrusted code safely (the sandbox)

`@wasm(fuel=N)` runs the kernel under a **fuel budget**: code that runs past the budget **traps**
instead of hanging your process. This is the point of running LLM-generated or untrusted code in
WASM — it's contained. (We compile it first with a harmless `spin(0)`, then let a runaway `spin(1)`
loop forever — the sandbox stops it. A plain-Python version of this call would hang the notebook.)

In [5]:
@wasm(fuel=5_000_000)
def spin(n: int) -> int:
    i = 0
    while n != 0:
        i = (i + 1) % 1000
    return i

spin(0)  # harmless (n == 0 returns at once); this first call compiles + binds the sandbox
assert binding_of(spin).mode == "server", "only run the runaway case if it is actually sandboxed"

try:
    spin(1)  # infinite loop -- but fuel-metered
    print("no trap (unexpected)")
except Exception as e:
    print(f"contained: {type(e).__name__} -- the sandbox stopped the runaway loop at the fuel budget")
    print("(a plain-Python spin(1) here would have hung forever)")

contained: FuelExhausted -- the sandbox stopped the runaway loop at the fuel budget
(a plain-Python spin(1) here would have hung forever)


## 4. 1-D arrays — pass your NumPy buffer, get `== NumPy`

An `@wasm` kernel can take typed array buffers directly (`Array[int32, 1]`, `float64`, `uint8`, ...).
You hand it a NumPy array; it fills an output buffer in place. The `from __future__ import
annotations` line keeps the `Array[...]` annotations lazy (that's the one requirement).

In [6]:
from __future__ import annotations
import numpy as np

@wasm
def scale_1d(a: Array[int32, 1], out: Array[int32, 1], k: int) -> int:
    n = len(a)
    for i in range(n):
        out[i] = a[i] * k
    return n

a32  = np.arange(1000, dtype=np.int32)
out  = np.zeros_like(a32)
scale_1d(a32, out, 7)

print("mode:", binding_of(scale_1d).mode)
print("== NumPy (bit-for-bit):", np.array_equal(out, a32 * 7))

mode: server
== NumPy (bit-for-bit): True


## 5. 2-D / image processing — the same thing the Gradio image demo runs

A 2-D `uint8` kernel: a nearest-neighbor image downscale on packed RGB rows (`Array[uint8, 2]`,
shape `[H, W*3]` — exactly a `rgb.reshape(H, W*3)`). This **is** the compute behind the
image-preprocess Gradio Space (resize in the browser before upload); here it runs in-process,
checked bit-for-bit against a NumPy reference. (Array rule: index elements as `img[y][x]`, and
pass `oh`/`ow` as params — there's no bare-row value in the array ABI.)

In [7]:
from __future__ import annotations

@wasm
def downscale_nn(img: Array[uint8, 2], scale: int, oh: int, ow: int, out: Array[uint8, 2]) -> int:
    for oy in range(oh):
        iy = oy * scale
        for ox in range(ow):
            ix = ox * scale * 3
            o = ox * 3
            out[oy][o] = img[iy][ix]
            out[oy][o + 1] = img[iy][ix + 1]
            out[oy][o + 2] = img[iy][ix + 2]
    return oh * ow

rng = np.random.RandomState(7)
H = W = 64
scale = 4
oh, ow = H // scale, W // scale
img = rng.randint(0, 256, size=(H, W * 3), dtype=np.uint8)   # packed RGB rows
out = np.zeros((oh, ow * 3), dtype=np.uint8)
downscale_nn(img, scale, oh, ow, out)

# NumPy nearest-neighbor reference: output pixel (oy, ox) = input pixel (oy*scale, ox*scale)
cols = (np.arange(ow)[:, None] * scale * 3 + np.arange(3)).ravel()
ref  = img[0:oh * scale:scale][:, cols]

print("mode:", binding_of(downscale_nn).mode)
print(f"downscaled {H}x{W} -> {oh}x{ow}")
print("== NumPy (bit-for-bit):", np.array_equal(out, ref))

mode: server
downscaled 64x64 -> 16x16
== NumPy (bit-for-bit): True


## Using the same `@wasm` function in Gradio / Streamlit

The adapters run that identical `@wasm` kernel **in the browser tab** (no server round-trip), or
in-process on the server — bit-for-bit the same as here. Compile-on-first-call applies here too:
the adapter's `dispatch(...)` compiles the kernel on first use, so there's **no explicit build
step**. A runnable version lives beside this notebook in **`gradio_demo.ipynb`**, which launches the app
in a browser tab from the notebook and records the timings; here's the real API surface:

```python
# app.py  --  run:  python app.py   (opens the Gradio UI; the .wasm runs in the browser tab)
from __future__ import annotations                 # keeps Array[...] annotations lazy
import gradio as gr
from pythscribe import wasm
from pythscribe.gradio import WasmFunction, dispatch, result_of

@wasm
def downscale_nn(img: Array[uint8, 2], scale: int, oh: int, ow: int, out: Array[uint8, 2]) -> int:
    ...   # the exact function from section 5

with gr.Blocks() as demo:
    comp = WasmFunction(label='downscale (@wasm)')  # the component that runs the .wasm in-tab
    # on a button/event:
    #   payload = dispatch(downscale_nn, img, scale, oh, ow, out)  # -> set as `comp`'s value
    #   answer  = result_of(payload, ...)                          # read the browser's result back
demo.launch()
```

Streamlit is the same idea via `pythscribe.streamlit`. Runnable examples live beside this notebook:
**`gradio_demo.ipynb`** launches this same kernel live in a Gradio tab (browser) and on the server,
timings and bit-for-bit fidelity side by side; **`full_features_demo.ipynb`** has the full use-case
tables. The point: you write the kernel **once**, `@wasm`, and it runs everywhere.

## Where `@wasm` helps — and where it doesn't

The speed win comes from tight, **non-vectorizable** loops (sections 1–2). For a single pass over a
large array — or already-vectorizable work — NumPy (and Numba) are faster, because crossing the data
boundary costs more than the loop saves. `@wasm` is **not** a blanket "make Python fast" button, and
it does not try to beat NumPy/Numba on raw numeric throughput.

What it **is**: the same ordinary Python function, running as one small sandboxed `.wasm` — in the
browser, in-process on the server (GIL-free, fuel-metered), in a Gradio / Streamlit app, or as plain
Python — **bit-for-bit identical**, with no Python runtime to ship and nothing to build by hand. You
just write `@wasm` and call the function.